In [3]:
import asyncio
from enum import IntEnum
from typing import Union, Any, Dict

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy, MultipleStructuredOutputsError
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.config import get_stream_writer
from pydantic import BaseModel, Field, computed_field
from rich import print as rprint
from config.config import ToolsModel

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    extra_body = {
        "thinking":{"type":"disabled"}
    }
)

In [4]:
class GetcurrentPromotions(BaseModel):
    pass

class QueryCustomerData(BaseModel):
    customer_id: str = Field(description="客户id,用于标识用户,唯一")

In [5]:
@tool(description="查询客户基本信息",args_schema=QueryCustomerData)
def query_customer_data(customer_id: str) -> Dict[str,Any]:
    return {"customer_id":customer_id,"name":"张三","level":"vip","join_data":"2023-01-15"}

@tool(description="获取当前可用促销活动",args_schema=GetcurrentPromotions)
def get_current_promotions() -> Dict[str,Any]:
    return {
        "promotions":["老用户优惠","会员专属折扣"],
        "valid_until":"2027-01-31"
    }

@tool(
    description="生成销售报告"
)
async def generate_sales_report() -> str:
    writer = get_stream_writer()

    writer({"type":"生成销售报告","message":"开始生成销售报告"})

    for i in range(1,4):
        await asyncio.sleep(0.5)
        writer({"type":"生成销售报告","message":f"生成消息报告进度: {i*25}%"})

    writer({"type":"生成销售报告","message":"报告生成完成"})

    return f"销售报告: 总收入160万,同比增长12%"

@tool(
    description="生成库存报告"
)
async def generate_inventory_report() -> str:
    writer = get_stream_writer()
    writer("开始库存分析...")
    await asyncio.sleep(0.5)
    writer("检查当前库存量...")
    await asyncio.sleep(0.5)
    writer("生成库存报告...")

    return f"当前库存量为10000件,库存充足,无异常。"


# Agent的高级用法-流式输出策略

## 1 values输出模式

```text
当stream_mode设置为values时,每个步骤执行后,都会输出完整状态信息,适用于每一步都要获取完整状态,持久化场景。
```

In [ ]:
agent = create_agent(
    name="agent高级用法-流式输出策略",
    model=model,
    tools=[get_current_promotions,query_customer_data]
)

resp = agent.stream(
    {
        "messages":[
            HumanMessage(content="查询客户id为1234的完整信息，和可用优惠")
        ]
    },
    stream_mode="values"
)

responses = []

for chunk in resp:
    responses.append(chunk)
    rprint(chunk)
    print("-" * 50)

final_resp = responses[-1]["messages"][-1].content

rprint(final_resp)

## 2 updates输出模式

```text
默认模式,执行步骤中,只增量更新状态发生变化的内容,用于监控Agent执行进度,例如观察Agent决定调用工具,工具执行结果等步骤
```

In [ ]:
agent = create_agent(
    name="agent高级用法-流式输出策略",
    model=model,
    tools=[get_current_promotions,query_customer_data]
)

resp = agent.stream(
    {
        "messages":[
            HumanMessage(content="查询客户id为1234的完整信息，和可用优惠")
        ]
    },
    stream_mode="updates"
)

responses = []

for chunk in resp:
    responses.append(chunk)
    rprint(chunk)
    print("-" * 50)

final_resp = responses[-1]["model"]["messages"][0].content

rprint(final_resp)

# 3 messages输出模式

```text
类大模型打字机形式，输出token等元信息
```

In [ ]:
agent = create_agent(
    name="agent高级用法-流式输出策略",
    model=model,
    tools=[get_current_promotions,query_customer_data]
)

resp = agent.stream(
    {
        "messages":[
            HumanMessage(content="查询客户id为1234的完整信息，和可用优惠")
        ]
    },
    stream_mode="messages"
)

for chunk in resp:
    print(chunk[0].content,end="",flush=True)


# 4 tasks输出模式

```text
输出task任务开始和结束时间,包含任务的结果和错误信息,用于监控任务的生命周期
```

In [ ]:
agent = create_agent(
    name="agent高级用法-流式输出策略",
    model=model,
    tools=[get_current_promotions,query_customer_data]
)

resp = agent.stream(
    {
        "messages":[
            HumanMessage(content="查询客户id为1234的完整信息，和可用优惠")
        ]
    },
    stream_mode="tasks"
)

for chunk in resp:
    rprint(chunk)
    print("-"*50)


# 5 debug输出模式

```text
与tasks模式类似,比tasks模式多输出任务步骤,时间戳,task类型(task/task_result),用于调试,监控task任务的生命周期。
```

In [ ]:
agent = create_agent(
    name="agent高级用法-流式输出策略",
    model=model,
    tools=[get_current_promotions,query_customer_data]
)

resp = agent.stream(
    {
        "messages":[
            HumanMessage(content="查询客户id为1234的完整信息，和可用优惠")
        ]
    },
    stream_mode="tasks"
)

for chunk in resp:
    rprint(chunk)
    print("-"*50)

# 6 checkpoints输出模式

```text
每当检查点(checkpoint)被创建时出发输出,输出包含检查点中的状态,用于需要持久化,工作流恢复或分布式执行跟踪的高级场景。
```

In [ ]:
# 创建内存检查点存储
checkpointer = InMemorySaver()

agent = create_agent(
    name="agent高级用法-流式输出策略",
    model=model,
    tools=[get_current_promotions,query_customer_data],
    checkpointer=checkpointer
)

# 创建唯一的会话id
config = {
    "configurable":{
        "thread_id":"session01"
    }
}

# 调用Agent
checkpoint_count = 0

resp = agent.stream(
    {
        "messages":[
            HumanMessage(content="查询客户id为1234的完整信息，和可用优惠")
        ]
    },
    stream_mode="checkpoints"
)

for chunk in resp:
    checkpoint_count += 1
    print(f"检查点 #{checkpoint_count}")
    rprint(chunk)
    print("-"*50)

## 7 custom输出模式

```text
通过get_stream_writer在工具或节点内部自定义发送的数据,用于输出业务逻辑相关的进度信息,自定义日志或指标。

```

In [6]:

async_agent = create_agent(
    name="异步agent01",
    model=model,
    tools=[generate_sales_report,generate_inventory_report]
)

async for chunk in async_agent.astream(
    {
        "messages":[
            HumanMessage(content="生成销售报告和库存报告")
        ]
    },
    stream_mode="custom"
):
    rprint(chunk)
    print("*"*50)

{'type': '生成销售报告', 'message': '开始生成销售报告'}

**************************************************


开始库存分析...

**************************************************


{'type': '生成销售报告', 'message': '生成消息报告进度: 25%'}

...

**************************************************


{'type': '生成销售报告', 'message': '生成消息报告进度: 75%'}

**************************************************


{'type': '生成销售报告', 'message': '报告生成完成'}

**************************************************
